# Processing Silver layer for account_account model

In [0]:
%sql
DESCRIBE bakery_pos_develop.bronze.account_account

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_account_account AS
SELECT
    id AS account_id,
    name AS account_name,
    account_type,
    CAST(create_date AS TIMESTAMP) AS created_at,
    CAST(write_date AS TIMESTAMP) AS updated_at,
    processing_timestamp AS bronze_ingested_at
FROM bakery_pos_develop.bronze.account_account
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY id
    ORDER BY write_date DESC,
    processing_timestamp DESC
) = 1
    

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bakery_pos_develop.silver.account_account (
    account_id INT,
    account_name STRING,
    account_type STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    bronze_ingested_at TIMESTAMP,
    silver_processed_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN
) USING DELTA;


In [0]:
%sql

MERGE INTO bakery_pos_develop.silver.account_account AS target
USING (
  SELECT *, current_timestamp() AS silver_processed_at 
  FROM stg_account_account
) AS source
ON target.account_id = source.account_id
WHEN MATCHED AND source.updated_at > target.updated_at THEN
  UPDATE SET
    account_name = source.account_name,
    account_type = source.account_type,
    created_at = source.created_at,
    updated_at = source.updated_at,
    is_updated = True,
    is_deleted = False,
    bronze_ingested_at = source.bronze_ingested_at,
    silver_processed_at = source.silver_processed_at
WHEN NOT MATCHED THEN 
  INSERT (
    account_id,
    account_name,
    account_type,
    created_at,
    updated_at,
    bronze_ingested_at,
    silver_processed_at,
    is_updated,
    is_deleted
  )
  VALUES (
    source.account_id,
    source.account_name,
    source.account_type,
    source.created_at,
    source.updated_at,
    source.bronze_ingested_at,
    source.silver_processed_at,
    False,
    False
  );

In [0]:
%sql

DESCRIBE bakery_pos_develop.silver.account_account